# Phase 3A: First Moment Incremental Clustering (FMIC)

US-3.19 runs independent FMIC monitoring at alpha 0.10 on the authoritative 
stable cohort. Mathematical definitions are shared with the preserved 
Phase 2-scheduled clustering foundation; the Phase 2 baseline is comparison-only.


In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

## Stage 1: Input Loading and Policy Resolution

This stage validates input artifacts, resolves preferred and fallback source policies, and normalizes numeric columns required by the deterministic state machine.

Run context note: repository paths resolve from `Path.cwd()` first and fall back to its parent when the notebook is executed from the `scripts` directory. If artifacts still are not found under the resolved project root, run the notebook from the expected repository root or adjust path resolution.

In [ ]:
# Stage 1: Load the aligned input, experiment config, and authoritative Phase 3A threshold.
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scripts").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = resolve_repo_root()
PHASE3A_CONFIG_PATH = REPO_ROOT / "config/phase3a_experiment_config.json"
if not PHASE3A_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Required Phase 3A config not found: {PHASE3A_CONFIG_PATH}")
phase3a_experiment_config = json.loads(
    PHASE3A_CONFIG_PATH.read_text(encoding="utf-8")
)
PHASE1_ALIGNED_PATH = REPO_ROOT / phase3a_experiment_config["input_source"]
PHASE1_STATS_PATH = REPO_ROOT / phase3a_experiment_config["threshold"]["statistics_artifact"]
PHASE2_OUTPUT_PATH = REPO_ROOT / "artifacts/phase2_temperature_sensors.csv"
PHASE2_METRICS_PATH = REPO_ROOT / "artifacts/phase2_temperature_metrics.json"
PHASE3A_THRESHOLD_SERIES_NAME = phase3a_experiment_config["threshold"]["source_series"]
PRIMARY_SOURCE_SERIES_NAME = PHASE3A_THRESHOLD_SERIES_NAME
PRIMARY_GLOBAL_AVG_COLUMN = "average_temperature_all_devices"
SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
GLOBAL_AVG_COLUMN = PRIMARY_GLOBAL_AVG_COLUMN


def require_input_files(paths):
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Required input not found: {path}")


def resolve_global_average_column(df):
    if PRIMARY_GLOBAL_AVG_COLUMN in df.columns:
        return PRIMARY_GLOBAL_AVG_COLUMN
    raise ValueError(
        "Could not find the required all-devices global average column in aligned phase1 data"
    )


def select_p90_row(stats_frame):
    row = stats_frame.loc[stats_frame["series_name"] == PHASE3A_THRESHOLD_SERIES_NAME]
    if len(row) != 1:
        raise ValueError(
            "Expected exactly one phase3a_stable_cohort row in phase1_average_statistics.csv"
        )
    return row


def find_sensor_columns(df):
    cols = sorted(
        [
            column
            for column in df.columns
            if column.endswith("_temperature")
            and column != PRIMARY_GLOBAL_AVG_COLUMN
        ]
    )
    if not cols:
        raise ValueError("No per-sensor temperature columns found in aligned phase1 data")
    return cols


require_input_files([PHASE1_ALIGNED_PATH, PHASE1_STATS_PATH])
phase1_df = pd.read_csv(PHASE1_ALIGNED_PATH, keep_default_na=False, na_values=[""])
stats_df = pd.read_csv(PHASE1_STATS_PATH, float_precision="round_trip")

if phase1_df.empty:
    raise ValueError("aligned_phase1_temperature.csv has no rows")

GLOBAL_AVG_COLUMN = resolve_global_average_column(phase1_df)
p90_row = select_p90_row(stats_df)
p90_threshold = float(p90_row.iloc[0]["p90"])
configured_p90 = phase3a_experiment_config["threshold"].get("p90_value")
if configured_p90 is None or p90_threshold != float(configured_p90):
    raise ValueError(
        "Phase 3A config P90 does not match the authoritative statistics row."
    )
sensor_cols = find_sensor_columns(phase1_df)

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts.phase2_monitoring import normalize_phase2_readings
phase1_df = normalize_phase2_readings(phase1_df, [GLOBAL_AVG_COLUMN, *sensor_cols])

print(
    json.dumps(
        {
            "repo_root": str(REPO_ROOT),
            "rows": int(len(phase1_df)),
            "sensor_count": int(len(sensor_cols)),
            "phase3a_p90_threshold": p90_threshold,
            "threshold_series": SOURCE_SERIES_NAME,
            "threshold_status": phase3a_experiment_config["threshold"]["status"],
            "phase3a_config": str(PHASE3A_CONFIG_PATH),
            "phase1_input": str(PHASE1_ALIGNED_PATH),
            "global_avg_column": GLOBAL_AVG_COLUMN,
        },
        indent=2,
    )
)

## Stage 2: History, Normalization, Clustering, and Input Validation

All Phase 3 core definitions live in this notebook. The tagged cell below retains the validated mathematics and is also loaded by separate automated tests.


In [ ]:
"""Source-aligned history normalization and clustering for Phase 3A."""

from __future__ import annotations

from dataclasses import dataclass
import math
from numbers import Real
from types import MappingProxyType
from typing import Iterable, Mapping

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class HistoryNormalizationSnapshot:
    """One event-level normalization result shared by alpha candidates."""

    eligible_sensor_ids: tuple[str, ...]
    raw_histories: Mapping[str, float]
    history_magnitudes: Mapping[str, float]
    maximum_history_magnitude: float | None
    normalized_histories: Mapping[str, float] | None
    normalization_status: str
    partition_basis: str
    zero_history_fallback: bool
    fallback_reason: str | None
    fallback_partition: tuple[tuple[str, ...], ...] | None
    deferred_reason: str | None


@dataclass(frozen=True)
class AlphaClusteringInput:
    """Per-alpha handoff referencing the shared normalization snapshot."""

    clustering_event_id: int
    synchronization_id: int
    alpha: float
    normalization_snapshot: HistoryNormalizationSnapshot


PHASE3_ALPHA_VALUES = tuple(value / 100 for value in range(5, 100, 5))
PHASE3A_DEFAULT_ALPHA_VALUES = (0.10,)


def normalize_sensor_histories(
    raw_histories: Mapping[str, Real],
    eligible_sensor_ids: Iterable[str],
) -> HistoryNormalizationSnapshot:
    """Normalize eligible scalar histories without mutating persistent state."""
    if isinstance(eligible_sensor_ids, str):
        raise ValueError("Provide eligible sensor IDs as an iterable, not a string.")
    eligible_ids = tuple(eligible_sensor_ids)
    if any(not isinstance(sensor_id, str) or not sensor_id for sensor_id in eligible_ids):
        raise ValueError("Eligible sensor IDs must be non-empty strings.")
    if len(set(eligible_ids)) != len(eligible_ids):
        raise ValueError("Eligible sensor IDs must be unique.")

    if not eligible_ids:
        return HistoryNormalizationSnapshot(
            eligible_sensor_ids=(),
            raw_histories=MappingProxyType({}),
            history_magnitudes=MappingProxyType({}),
            maximum_history_magnitude=None,
            normalized_histories=None,
            normalization_status="deferred",
            partition_basis="not_applicable",
            zero_history_fallback=False,
            fallback_reason=None,
            fallback_partition=None,
            deferred_reason="no_eligible_sensors",
        )

    missing = sorted(set(eligible_ids) - set(raw_histories))
    if missing:
        raise ValueError(f"Raw history is missing eligible sensors: {missing}.")

    eligible_histories: dict[str, float] = {}
    for sensor_id in sorted(eligible_ids):
        value = raw_histories[sensor_id]
        if isinstance(value, bool) or not isinstance(value, Real):
            raise ValueError(f"History for {sensor_id} must be a finite number.")
        numeric_value = float(value)
        if not math.isfinite(numeric_value):
            raise ValueError(f"History for {sensor_id} must be finite.")
        eligible_histories[sensor_id] = numeric_value

    magnitudes = {
        sensor_id: abs(history)
        for sensor_id, history in eligible_histories.items()
    }
    maximum_magnitude = max(magnitudes.values())
    if maximum_magnitude == 0:
        return HistoryNormalizationSnapshot(
            eligible_sensor_ids=tuple(sorted(eligible_histories)),
            raw_histories=MappingProxyType(eligible_histories),
            history_magnitudes=MappingProxyType(magnitudes),
            maximum_history_magnitude=0.0,
            normalized_histories=None,
            normalization_status="all_zero_history_fallback",
            partition_basis="all_zero_history_fallback",
            zero_history_fallback=True,
            fallback_reason="all_eligible_histories_are_zero",
            fallback_partition=tuple(
                (sensor_id,) for sensor_id in sorted(eligible_histories)
            ),
            deferred_reason=None,
        )

    normalized = {
        sensor_id: history / maximum_magnitude
        for sensor_id, history in eligible_histories.items()
    }
    return HistoryNormalizationSnapshot(
        eligible_sensor_ids=tuple(sorted(eligible_histories)),
        raw_histories=MappingProxyType(eligible_histories),
        history_magnitudes=MappingProxyType(magnitudes),
        maximum_history_magnitude=maximum_magnitude,
        normalized_histories=MappingProxyType(normalized),
        normalization_status="normalized",
        partition_basis="normalized_history",
        zero_history_fallback=False,
        fallback_reason=None,
        fallback_partition=None,
        deferred_reason=None,
    )


def build_alpha_clustering_inputs(
    synchronization_id: int,
    normalization_snapshot: HistoryNormalizationSnapshot,
    alpha_values: Iterable[Real] = PHASE3A_DEFAULT_ALPHA_VALUES,
) -> tuple[AlphaClusteringInput, ...]:
    """Build selected alpha inputs, sharing one event normalization result."""
    if synchronization_id < 1:
        raise ValueError("Synchronization ID must be positive.")
    if normalization_snapshot.deferred_reason is not None:
        return ()
    return tuple(
        AlphaClusteringInput(
            clustering_event_id=synchronization_id,
            synchronization_id=synchronization_id,
            alpha=float(alpha),
            normalization_snapshot=normalization_snapshot,
        )
        for alpha in alpha_values
    )


Cluster = tuple[str, ...]
Partition = tuple[Cluster, ...]


def _validated_clustering_values(
    normalized_histories: Mapping[str, Real], alpha: Real,
) -> dict[str, float]:
    if isinstance(alpha, bool) or not isinstance(alpha, Real):
        raise ValueError("Alpha must be a finite number in [0, 1].")
    if not math.isfinite(float(alpha)) or not 0 <= alpha <= 1:
        raise ValueError("Alpha must be a finite number in [0, 1].")
    values: dict[str, float] = {}
    for sensor_id, value in normalized_histories.items():
        if not isinstance(sensor_id, str) or not sensor_id:
            raise ValueError("Clustering sensor IDs must be non-empty strings.")
        if isinstance(value, bool) or not isinstance(value, Real):
            raise ValueError(f"Normalized history for {sensor_id} must be finite.")
        numeric_value = float(value)
        if not math.isfinite(numeric_value) or abs(numeric_value) > 1:
            raise ValueError(f"Normalized history for {sensor_id} must be in [-1, 1].")
        values[sensor_id] = numeric_value
    return values


def _canonical_cluster(cluster: Iterable[str]) -> Cluster:
    if isinstance(cluster, str):
        raise ValueError("A cluster must be an iterable of sensor IDs, not a string.")
    members = tuple(cluster)
    if not members or any(not isinstance(member, str) or not member for member in members):
        raise ValueError("Clusters must contain non-empty string sensor IDs.")
    if len(set(members)) != len(members):
        raise ValueError("A cluster must not contain duplicate sensor IDs.")
    return tuple(sorted(members))


def _cluster_quality(cluster: Cluster, values: Mapping[str, float], alpha: float) -> float:
    return abs(math.fsum(values[sensor_id] for sensor_id in cluster)) / len(cluster) + alpha * len(cluster)


def cluster_quality(
    cluster: Iterable[str], normalized_histories: Mapping[str, Real], alpha: Real,
) -> float:
    """Score signed cancellation before applying the scalar absolute value."""
    values = _validated_clustering_values(normalized_histories, alpha)
    members = _canonical_cluster(cluster)
    unknown = sorted(set(members) - set(values))
    if unknown:
        raise ValueError(f"Cluster contains sensors without normalized histories: {unknown}.")
    return _cluster_quality(members, values, float(alpha))


def partition_quality(
    partition: Iterable[Iterable[str]], normalized_histories: Mapping[str, Real], alpha: Real,
) -> float:
    """Score a disjoint partition containing exactly the supplied sensors."""
    values = _validated_clustering_values(normalized_histories, alpha)
    clusters = tuple(sorted(_canonical_cluster(cluster) for cluster in partition))
    members = tuple(member for cluster in clusters for member in cluster)
    if not clusters or len(set(members)) != len(members) or set(members) != set(values):
        raise ValueError("Partition must cover every eligible sensor exactly once.")
    return max(_cluster_quality(cluster, values, float(alpha)) for cluster in clusters)


def evaluate_alpha_clustering(clustering_input: AlphaClusteringInput) -> dict[str, object] | None:
    """Retain the greedy candidate sequence and choose its earliest minimum."""
    snapshot = clustering_input.normalization_snapshot
    if snapshot.deferred_reason is not None:
        return None
    eligible = snapshot.eligible_sensor_ids
    if not eligible:
        raise ValueError("A completed clustering input must have eligible sensors.")
    if snapshot.zero_history_fallback:
        if any(snapshot.raw_histories[sensor_id] != 0 for sensor_id in eligible):
            raise ValueError("All-zero fallback requires zero raw eligible histories.")
        # Zero raw histories allow fallback scoring without fabricating normalization.
        source_values = snapshot.raw_histories
    else:
        if snapshot.normalized_histories is None:
            raise ValueError("Nonzero history clustering requires normalized histories.")
        source_values = snapshot.normalized_histories
    values = _validated_clustering_values(source_values, clustering_input.alpha)
    if len(set(eligible)) != len(eligible) or set(values) != set(eligible):
        raise ValueError("Clustering values must cover exactly the eligible sensor set.")
    alpha = float(clustering_input.alpha)
    partition: Partition = tuple((sensor_id,) for sensor_id in sorted(eligible))
    candidates: list[dict[str, object]] = []
    candidate_qualities: list[float] = []
    candidate_partitions: list[Partition] = []
    merges: list[dict[str, object]] = []

    while True:
        qualities = {cluster: _cluster_quality(cluster, values, alpha) for cluster in partition}
        quality = max(qualities.values())
        candidates.append({
            "candidate_index": len(candidates),
            "partition": partition,
            "cluster_count": len(partition),
            "cluster_qualities": [
                {"members": cluster, "quality": qualities[cluster]} for cluster in partition
            ],
            "partition_quality": quality,
        })
        candidate_qualities.append(quality)
        candidate_partitions.append(partition)
        if snapshot.zero_history_fallback or len(partition) <= 2:
            break

        worst = min(partition, key=lambda cluster: (-qualities[cluster], cluster))
        partners = []
        for partner in partition:
            if partner != worst:
                merged = tuple(sorted(worst + partner))
                partners.append((partner, merged, _cluster_quality(merged, values, alpha)))
        partner, merged, merged_quality = min(partners, key=lambda item: (item[2], item[0]))
        merges.append({
            "from_candidate_index": len(candidates) - 1,
            "worst_cluster": worst,
            "worst_cluster_quality": qualities[worst],
            "partner_scores": [
                {"partner": item[0], "merged_cluster": item[1], "merged_quality": item[2]}
                for item in partners
            ],
            "selected_partner": partner,
            "merged_cluster": merged,
            "merged_quality": merged_quality,
        })
        partition = tuple(sorted(
            tuple(cluster for cluster in partition if cluster not in (worst, partner)) + (merged,)
        ))

    selected_index = min(range(len(candidates)), key=lambda index: candidate_qualities[index])
    selected_partition = candidate_partitions[selected_index]
    return {
        "clustering_event_id": clustering_input.clustering_event_id,
        "synchronization_id": clustering_input.synchronization_id,
        "normalization_snapshot_id": clustering_input.synchronization_id,
        "alpha": alpha,
        "partition_basis": snapshot.partition_basis,
        "zero_history_fallback": snapshot.zero_history_fallback,
        "fallback_reason": snapshot.fallback_reason,
        "candidate_partitions": candidates,
        "merge_steps": merges,
        "selected_candidate_index": selected_index,
        "selected_partition": selected_partition,
        "partition_quality": candidate_qualities[selected_index],
        "cluster_count": len(selected_partition),
        "cluster_assignments": {
            sensor_id: cluster_index
            for cluster_index, cluster in enumerate(selected_partition)
            for sensor_id in cluster
        },
    }

PHASE3A_APPROVED_SENSOR_IDS = ("A", "C", "D", "E", "F", "G")
PHASE3A_BUCKET_SECONDS = 60


def validate_phase3a_config(config):
    """Reject configuration drift from the approved stable-cohort contract."""
    if not isinstance(config, dict):
        raise ValueError("Phase 3A experiment configuration must be a JSON object.")
    if config.get("input_source") != "artifacts/aligned_phase1_temperature.csv":
        raise ValueError("Phase 3A input must use the aligned Phase 1 artifact.")

    sensor_ids = tuple(config.get("selected_sensor_ids", ()))
    if sensor_ids != PHASE3A_APPROVED_SENSOR_IDS:
        raise ValueError(
            "Phase 3A cohort must remain exactly A, C, D, E, F, and G."
        )
    expected_columns = [f"{sensor_id}_temperature" for sensor_id in sensor_ids]
    if config.get("selected_sensor_temperature_columns") != expected_columns:
        raise ValueError(
            "Phase 3A temperature columns must match the approved sensor cohort."
        )
    if config.get("endpoint_convention") != "inclusive":
        raise ValueError("Phase 3A interval endpoints must both be inclusive.")
    if config.get("sampling_interval_seconds") != PHASE3A_BUCKET_SECONDS:
        raise ValueError("Phase 3A sampling interval must be exactly 60 seconds.")
    if config.get("stable_cohort_required") is not True:
        raise ValueError("Phase 3A must require the fixed stable cohort.")
    if config.get("missing_value_policy") != "fail":
        raise ValueError("Phase 3A missing observations must fail validation.")
    if config.get("imputation_policy") != "none":
        raise ValueError("Phase 3A input must not use imputed sensor values.")
    if config.get("baseline_alpha") != 0.10:
        raise ValueError("The Phase 3A baseline alpha must be 0.10.")
    if config.get("boundary_convention") != {
        "safe_when": "metric < threshold",
        "violated_when": "metric >= threshold",
    }:
        raise ValueError("Phase 3A boundary convention does not match the approved policy.")

    threshold_config = config.get("threshold")
    if not isinstance(threshold_config, dict):
        raise ValueError("Phase 3A threshold configuration must be an object.")
    if threshold_config.get("source_series") != "phase3a_stable_cohort":
        raise ValueError("The Phase 3A threshold source must be phase3a_stable_cohort.")
    if threshold_config.get("statistic") != "p90":
        raise ValueError("The authoritative Phase 3A threshold statistic must be p90.")
    threshold_value = threshold_config.get("p90_value")
    if isinstance(threshold_value, bool) or not isinstance(
        threshold_value, (int, float)
    ) or not np.isfinite(threshold_value):
        raise ValueError("The authoritative Phase 3A P90 must be a finite number.")
    if threshold_config.get("status") != "authoritative":
        raise ValueError("The Phase 3A P90 must be marked authoritative.")
    if threshold_config.get("statistics_artifact") != "artifacts/phase1_average_statistics.csv":
        raise ValueError("Phase 3A P90 provenance must name the Phase 1 statistics artifact.")
    if threshold_config.get("production_script") != "scripts/build_phase1_temperature.py":
        raise ValueError("Phase 3A P90 provenance must name the Phase 1 production script.")

    start_time = pd.Timestamp(config.get("start_timestamp_utc"))
    end_time = pd.Timestamp(config.get("end_timestamp_utc"))
    if start_time.tz is None or end_time.tz is None:
        raise ValueError("Phase 3A interval timestamps must be timezone-aware.")
    start_time = start_time.tz_convert("UTC")
    end_time = end_time.tz_convert("UTC")
    if end_time <= start_time:
        raise ValueError("Phase 3A end timestamp must follow its start timestamp.")

    elapsed_seconds = (end_time - start_time).total_seconds()
    if elapsed_seconds % PHASE3A_BUCKET_SECONDS:
        raise ValueError("Phase 3A endpoints must align to one-minute buckets.")
    expected_elapsed_minutes = int(elapsed_seconds // 60)
    expected_bucket_count = config.get("expected_bucket_count")
    if isinstance(expected_bucket_count, bool) or not isinstance(
        expected_bucket_count, int
    ):
        raise ValueError("Phase 3A expected_bucket_count must be an integer.")
    if expected_bucket_count != expected_elapsed_minutes + 1:
        raise ValueError("Phase 3A bucket count must include both interval endpoints.")
    if config.get("expected_elapsed_minutes") != expected_elapsed_minutes:
        raise ValueError("Phase 3A expected elapsed minutes do not match its endpoints.")

    return sensor_ids, expected_columns, start_time, end_time, expected_bucket_count


def prepare_phase3a_input(source_frame, experiment_config):
    """Select and validate the approved complete one-minute Phase 3A input."""
    sensor_ids, sensor_columns, start_time, end_time, expected_bucket_count = (
        validate_phase3a_config(experiment_config)
    )
    required_columns = ["bucket_epoch", "bucket_time_utc", *sensor_columns]
    missing_columns = sorted(set(required_columns) - set(source_frame.columns))
    if missing_columns:
        raise ValueError(f"Phase 3A input is missing required columns: {missing_columns}")

    timestamps = pd.to_datetime(
        source_frame["bucket_time_utc"], utc=True, errors="coerce"
    )
    if timestamps.isna().any():
        raise ValueError("Phase 1 input contains an invalid bucket_time_utc timestamp.")
    in_interval = timestamps.between(start_time, end_time, inclusive="both")
    selected = source_frame.loc[in_interval, required_columns].copy()
    selected["bucket_time_utc"] = pd.to_datetime(
        selected["bucket_time_utc"], utc=True, errors="raise"
    )

    if len(selected) != expected_bucket_count:
        raise ValueError(
            f"Phase 3A interval requires {expected_bucket_count} buckets; "
            f"found {len(selected)}."
        )
    selected_times = selected["bucket_time_utc"]
    if not selected_times.is_unique:
        raise ValueError("Phase 3A input contains duplicate timestamps.")
    if not selected_times.is_monotonic_increasing:
        raise ValueError("Phase 3A timestamps must be in strictly increasing order.")
    if selected_times.iloc[0] != start_time:
        raise ValueError("Phase 3A first timestamp does not match the approved start.")
    if selected_times.iloc[-1] != end_time:
        raise ValueError("Phase 3A final timestamp does not match the approved end.")

    time_deltas = selected_times.diff().dt.total_seconds().iloc[1:]
    if not time_deltas.eq(PHASE3A_BUCKET_SECONDS).all():
        bad_position = int(
            np.flatnonzero(
                ~time_deltas.eq(PHASE3A_BUCKET_SECONDS).to_numpy()
            )[0]
        ) + 1
        raise ValueError(
            f"Phase 3A timestamp gap at selected row {bad_position}; "
            "expected exactly 60 seconds."
        )

    epochs = pd.to_numeric(selected["bucket_epoch"], errors="coerce")
    epoch_values = epochs.to_numpy(dtype=float)
    if not np.isfinite(epoch_values).all() or not np.equal(
        epoch_values, np.floor(epoch_values)
    ).all():
        raise ValueError("Phase 3A bucket_epoch values must be finite whole seconds.")
    expected_epochs = np.fromiter(
        (int(timestamp.timestamp()) for timestamp in selected_times),
        dtype=np.int64,
        count=len(selected_times),
    )
    if not np.array_equal(epochs.to_numpy(dtype=np.int64), expected_epochs):
        raise ValueError("Phase 3A bucket_epoch values do not match timestamps.")
    if not epochs.diff().iloc[1:].eq(PHASE3A_BUCKET_SECONDS).all():
        raise ValueError("Phase 3A bucket_epoch values must be consecutive 60-second steps.")
    selected["bucket_epoch"] = epochs.astype("int64")

    for column in sensor_columns:
        numeric = pd.to_numeric(selected[column], errors="coerce")
        invalid = numeric.isna() | ~np.isfinite(numeric.to_numpy(dtype=float))
        if invalid.any():
            row_number = int(np.flatnonzero(invalid.to_numpy())[0])
            raise ValueError(
                f"Phase 3A requires a finite {column} value in every bucket; "
                f"missing or invalid value at {selected_times.iloc[row_number]}."
            )
        selected[column] = numeric.astype(float)
    return selected.reset_index(drop=True)


class Phase3AHistoryTracker:
    """Track signed accumulated history for an always-present fixed cohort."""

    DISCOUNT = 0.5

    def __init__(self, sensor_ids):
        self.sensor_ids = tuple(sensor_ids)
        if not self.sensor_ids or len(set(self.sensor_ids)) != len(self.sensor_ids):
            raise ValueError("Phase 3A sensor IDs must be non-empty and unique.")
        self.histories = {sensor_id: 0.0 for sensor_id in self.sensor_ids}
        self.references = {sensor_id: None for sensor_id in self.sensor_ids}
        self.synchronization_id = 0
        self.row_number = 0

    def step(self, readings, synchronization_causes=()):
        if set(readings) != set(self.sensor_ids):
            raise ValueError("Phase 3A readings must contain exactly the fixed cohort.")
        values = {}
        for sensor_id in self.sensor_ids:
            value = readings[sensor_id]
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number)):
                raise ValueError(f"Phase 3A reading for {sensor_id} must be finite and numeric.")
            value = float(value)
            if not np.isfinite(value):
                raise ValueError(f"Phase 3A reading for {sensor_id} must be finite and numeric.")
            values[sensor_id] = value

        if isinstance(synchronization_causes, str):
            raise ValueError("Synchronization causes must be an iterable, not a string.")
        causes = tuple(sorted(set(synchronization_causes)))
        if self.synchronization_id == 0 and not causes:
            raise ValueError("The first Phase 3A row must complete initialization synchronization.")
        if self.synchronization_id:
            for sensor_id in self.sensor_ids:
                self.histories[sensor_id] = (
                    self.DISCOUNT * self.histories[sensor_id]
                    + values[sensor_id] - self.references[sensor_id]
                )
        clustering_histories = None
        if causes:
            self.synchronization_id += 1
            clustering_histories = dict(self.histories)
            self.references = dict(values)
        self.row_number += 1
        return clustering_histories, causes



## Stage 3: FMIC Allocation and Monitoring

Source-defined first moments, the labeled project zero-weight fallback, and the shared node/coordinator/root engine. Import-only tests load this tagged cell after the mathematical definitions; no production run occurs here.


In [ ]:
"""Independent FMIC monitoring for the fixed Phase 3A experiment.

Chapter section 1.4, printed p.10 (PDF p.14): mu is the arithmetic
mean of the discounted vectors {v(t_m), v(t_(m-1))/2, ...}.
FMIC keeps only |mu| in the general moment weight (section 1.5.2).
Clustering history and the first moment are deliberately separate states.
"""

from __future__ import annotations

import hashlib
import json
import math
from collections.abc import Callable, Mapping, Sequence
from numbers import Real
from pathlib import Path

import pandas as pd


MESSAGE_CATEGORIES = (
    "node_trigger_messages",
    "cluster_check_requests",
    "cluster_check_responses",
    "coordinator_root_escalations",
    "root_sync_requests",
    "root_sync_responses",
    "root_state_updates",
)


def fmic_allocation(
    moments: Mapping[str, float],
    partition: Sequence[Sequence[str]],
    root_allowance: float,
) -> dict:
    if not moments or any(
        isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value)
        for value in moments.values()
    ):
        raise ValueError("FMIC moments must be nonempty and finite.")
    if not math.isfinite(root_allowance) or root_allowance < 0:
        raise ValueError("Root allowance must be finite and nonnegative.")
    members = [sensor for cluster in partition for sensor in cluster]
    if any(not cluster for cluster in partition) or sorted(members) != sorted(moments):
        raise ValueError("Partition must cover every sensor exactly once.")
    weights = {sensor: abs(value) for sensor, value in moments.items()}
    total = math.fsum(weights.values())
    if not math.isfinite(total):
        raise ValueError("FMIC total weight overflowed.")
    fallback = total == 0
    scaled = {
        sensor: 1.0 if fallback else len(weights) * (value / total)
        for sensor, value in weights.items()
    }
    node = {sensor: value * root_allowance for sensor, value in scaled.items()}
    if any(not math.isfinite(value) for value in node.values()):
        raise ValueError("FMIC node allowances overflowed.")
    coordinator = {
        max(cluster): math.fsum(node[sensor] for sensor in cluster) / len(cluster)
        for cluster in partition
    }
    return {
        "first_moments": dict(moments),
        "raw_weights": weights,
        "scaled_weights": scaled,
        "node_allowances": node,
        "coordinator_allowances": coordinator,
        "weight_status": "project_all_zero_equal_weight_fallback" if fallback else "source_first_moment",
    }


class DiscountedFirstMoment:
    """Source V' contains all post-initialization readings, discounted by age."""

    def __init__(self, sensors: Sequence[str]):
        self.sums = dict.fromkeys(sensors, 0.0)
        self.count = 0

    def update(self, readings: Mapping[str, float]) -> None:
        if set(readings) != set(self.sums):
            raise ValueError("First moments require the fixed sensor cohort.")
        if any(
            isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value)
            for value in readings.values()
        ):
            raise ValueError("First-moment observations must be finite.")
        self.count += 1
        self.sums = {
            sensor: 0.5 * self.sums[sensor] + readings[sensor]
            for sensor in self.sums
        }
        if any(not math.isfinite(value) for value in self.sums.values()):
            raise ValueError("Discounted first-moment sums overflowed.")

    def means(self) -> dict[str, float]:
        return {
            sensor: value / self.count if self.count else 0.0
            for sensor, value in self.sums.items()
        }


class FMICMonitor:
    """One deterministic node/coordinator/root protocol; no Phase 2 schedule."""

    def __init__(
        self,
        sensor_ids: Sequence[str],
        threshold: float,
        alpha: float = 0.10,
        *,
        clustering: Callable = evaluate_alpha_clustering,
    ):
        self.sensors = tuple(sensor_ids)
        if not self.sensors or len(set(self.sensors)) != len(self.sensors):
            raise ValueError("Sensors must be nonempty and unique.")
        if not math.isfinite(threshold) or not math.isfinite(alpha) or not 0 <= alpha <= 1:
            raise ValueError("Threshold and alpha must be finite; alpha must be in [0, 1].")
        self.threshold = threshold
        self.alpha = alpha
        self.clustering = clustering
        self.history = Phase3AHistoryTracker(self.sensors)
        self.moment = DiscountedFirstMoment(self.sensors)
        self.references: dict[str, float] = {}
        self.partition: tuple[tuple[str, ...], ...] = ()
        self.allocation: dict = {}
        self.known_violated = False
        self.events: list[dict] = []
        self.messages = dict.fromkeys(MESSAGE_CATEGORIES, 0)
        self.rows = 0
        self.node_triggers = 0
        self.checks = 0
        self.escalations = 0
        self.truth_violations = 0
        self.false_safe = 0
        self.false_violated = 0

    def step(self, readings: Mapping[str, float], timestamp: str, epoch: int) -> dict:
        if set(readings) != set(self.sensors):
            raise ValueError("FMIC readings must contain exactly the fixed cohort.")
        if any(isinstance(value, bool) or not isinstance(value, Real) for value in readings.values()):
            raise ValueError("FMIC observations must be numeric, not booleans or text.")
        values = {sensor: float(readings[sensor]) for sensor in self.sensors}
        if any(not math.isfinite(value) for value in values.values()):
            raise ValueError("FMIC observations must be finite.")
        counts = dict.fromkeys(MESSAGE_CATEGORIES, 0)
        initialization = not self.references
        check_count = escalation_count = trigger_count = 0
        if not initialization:
            self.moment.update(values)
            for cluster in self.partition:
                coordinator = max(cluster)
                deviations = {
                    sensor: values[sensor] - self.references[sensor] for sensor in cluster
                }
                triggers = {
                    sensor for sensor in cluster
                    if abs(deviations[sensor]) >= self.allocation["node_allowances"][sensor]
                }
                if not triggers:
                    continue
                trigger_count += len(triggers)
                check_count += 1
                # All triggering non-coordinators send their current reading.
                # The coordinator requests only peers not already reporting.
                counts["node_trigger_messages"] += len(triggers - {coordinator})
                requested = set(cluster) - triggers - {coordinator}
                counts["cluster_check_requests"] += len(requested)
                counts["cluster_check_responses"] += len(requested)
                cluster_change = abs(math.fsum(deviations.values()) / len(cluster))
                if cluster_change >= self.allocation["coordinator_allowances"][coordinator]:
                    escalation_count += 1
                    counts["coordinator_root_escalations"] += 1
        synchronize = initialization or escalation_count > 0
        causes = ("initialization",) if initialization else (
            ("coordinator_violation",) if synchronize else ()
        )
        histories, _ = self.history.step(values, synchronization_causes=causes)
        actual_mean = math.fsum(values.values()) / len(values)
        actual_violated = actual_mean >= self.threshold
        if synchronize:
            # The separate root has no sensor readings. Alerts carry no raw
            # readings; one collection obtains every sensor's state directly.
            counts["root_sync_requests"] = len(self.sensors)
            counts["root_sync_responses"] = len(self.sensors)
            counts["root_state_updates"] = len(self.sensors)
            if histories is None:
                raise RuntimeError("Synchronization did not produce a history snapshot.")
            snapshot = normalize_sensor_histories(histories, self.sensors)
            evaluation = self.clustering(
                AlphaClusteringInput(
                    self.history.synchronization_id,
                    self.history.synchronization_id, self.alpha, snapshot,
                )
            )
            if evaluation is None:
                raise RuntimeError("FMIC synchronization unexpectedly deferred clustering.")
            self.partition = tuple(tuple(cluster) for cluster in evaluation["selected_partition"])
            root_allowance = abs(self.threshold - actual_mean)
            self.allocation = fmic_allocation(
                self.moment.means(), self.partition, root_allowance
            )
            self.references = dict(values)
            self.known_violated = actual_violated
            event = {
                "clustering_event_id": self.history.synchronization_id,
                "synchronization_id": self.history.synchronization_id,
                "bucket_epoch": epoch, "bucket_time_utc": timestamp,
                "algorithm": "FMIC", "alpha": self.alpha,
                "synchronization_causes": "|".join(causes),
                "raw_sensor_histories": json.dumps(histories, sort_keys=True),
                "normalized_histories": json.dumps(
                    dict(snapshot.normalized_histories)
                    if snapshot.normalized_histories is not None else None, sort_keys=True,
                ),
                "normalization_status": snapshot.normalization_status,
                "selected_partition": json.dumps(self.partition),
                "selected_partition_quality": evaluation["partition_quality"],
                "cluster_count": len(self.partition),
                "coordinator_assignments": json.dumps(
                    {str(i): max(cluster) for i, cluster in enumerate(self.partition)}
                ),
                "coordinator_rule": "lexicographically_highest_member",
                "global_average": actual_mean, "global_violated": int(actual_violated),
                "p90_threshold": self.threshold, "root_allowance": root_allowance,
                "moment_observation_count": self.moment.count,
                "weight_status": self.allocation["weight_status"],
            }
            for field in (
                "first_moments", "raw_weights", "scaled_weights",
                "node_allowances", "coordinator_allowances",
            ):
                event[field] = json.dumps(self.allocation[field], sort_keys=True)
            self.events.append(event)
        false_safe = actual_violated and not self.known_violated
        self.rows += 1
        self.truth_violations += int(actual_violated)
        self.false_safe += int(false_safe)
        self.false_violated += int(not actual_violated and self.known_violated)
        self.node_triggers += trigger_count
        self.checks += check_count
        self.escalations += escalation_count
        for category in counts:
            self.messages[category] += counts[category]
        if false_safe:
            raise RuntimeError(f"Blocking FMIC false-safe at {timestamp}.")
        return {
            "synchronized": synchronize, "global_violated": actual_violated,
            "known_violated": self.known_violated, "false_safe": false_safe,
            "node_triggers": trigger_count, "coordinator_checks": check_count,
            "coordinator_escalations": escalation_count, "messages": counts,
            "total_messages": sum(counts.values()),
        }

    def summary(self) -> dict:
        if not self.rows:
            raise ValueError("Cannot summarize an uninitialized FMIC monitor.")
        cluster_sizes = [
            len(cluster) for event in self.events
            for cluster in json.loads(event["selected_partition"])
        ]
        return {
            "algorithm": "FMIC", "alpha": self.alpha,
            "bucket_count": self.rows,
            "total_messages": sum(self.messages.values()),
            "message_category_totals": dict(self.messages),
            "root_synchronization_count": len(self.events),
            "initialization_count": 1,
            "monitoring_triggered_synchronization_count": len(self.events) - 1,
            "node_trigger_count": self.node_triggers,
            "coordinator_check_count": self.checks,
            "coordinator_to_root_escalation_count": self.escalations,
            "global_threshold_violation_count": self.truth_violations,
            "false_safe_count": self.false_safe,
            "false_violated_count": self.false_violated,
            "correctness_status": "passed" if self.false_safe == 0 else "failed",
            "message_counts_reconcile": True,
            "cluster_count_min": min(event["cluster_count"] for event in self.events),
            "cluster_count_max": max(event["cluster_count"] for event in self.events),
            "cluster_size_min": min(cluster_sizes),
            "cluster_size_max": max(cluster_sizes),
            "cluster_size_mean_across_sync_clusters": math.fsum(cluster_sizes) / len(cluster_sizes),
            "root_model": "separate_root_component",
            "monitoring_policy": "absolute distance to threshold; root-known state retained until sync",
            "moment_policy": "all post-initialization raw readings; discounted_sum = 0.5 * prior_sum + reading; mu = discounted_sum / sample_count; never reset at sync",
            "weight_zero_policy": "project fallback: all W_n=0 gives w_n=1; individual zero weights remain zero",
            "protocol": "all clusters checked once per bucket; all triggering peers report; collect remaining peers; alerts carry no raw readings; one root collection per bucket; updates take effect next bucket",
            "correctness_policy": "centralized truth is validation-only and never triggers communication",
            "source": "Clustering for Monitoring Distributed Data Streams, section 1.4 p.10-11, section 1.5.2; separate root and >= boundary are project conventions",
        }


def run_phase3a_fmic(root: Path) -> dict:
    config_path = root / "config" / "phase3a_experiment_config.json"
    config = json.loads(config_path.read_text(encoding="utf-8"))
    sensors, _, _, _, _ = validate_phase3a_config(config)
    if config["baseline_alpha"] != 0.10:
        raise ValueError("US-3.19 production FMIC requires alpha=0.10.")
    threshold_config = config["threshold"]
    threshold = float(threshold_config["p90_value"])
    statistics = pd.read_csv(
        root / threshold_config["statistics_artifact"], float_precision="round_trip"
    )
    threshold_rows = statistics.loc[
        statistics["series_name"] == threshold_config["source_series"], "p90"
    ]
    if len(threshold_rows) != 1 or float(threshold_rows.iloc[0]) != threshold:
        raise ValueError("FMIC threshold does not match the authoritative statistics artifact.")
    input_path = root / config["input_source"]
    frame = pd.read_csv(input_path, float_precision="round_trip")
    selected = prepare_phase3a_input(frame, config)
    baseline_path = root / "artifacts" / "phase3a_phase2_baseline.json"
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    for key in (
        "experiment_name", "input_source", "selected_sensor_ids", "start_timestamp_utc",
        "end_timestamp_utc", "endpoint_convention", "boundary_convention",
    ):
        if baseline[key] != config[key]:
            raise ValueError(f"Phase 2 baseline does not match FMIC {key}.")
    if (
        baseline["bucket_count"] != len(selected)
        or baseline["threshold_value"] != threshold
        or baseline["threshold_source_series"] != threshold_config["source_series"]
        or baseline["correctness_status"] != "passed"
        or baseline["algorithm"] != "phase2"
        or sum(baseline["message_category_totals"].values()) != baseline["total_messages"]
    ):
        raise ValueError("Invalid matched Phase 2 baseline for FMIC comparison.")
    monitor = FMICMonitor(sensors, threshold, config["baseline_alpha"])
    for row in selected.to_dict("records"):
        monitor.step(
            {sensor: row[f"{sensor}_temperature"] for sensor in sensors},
            row["bucket_time_utc"].isoformat(), int(row["bucket_epoch"]),
        )
    summary = monitor.summary()
    if summary["global_threshold_violation_count"] != baseline["global_violation_count"]:
        raise ValueError("FMIC centralized truth does not match the Phase 2 baseline.")
    summary.update({
        key: config[key] for key in (
            "experiment_name", "input_source", "selected_sensor_ids",
            "start_timestamp_utc", "end_timestamp_utc", "endpoint_convention",
            "boundary_convention",
        )
    })
    summary.update({
        "threshold_source_series": threshold_config["source_series"],
        "threshold_value": threshold,
        "threshold_statistic": threshold_config["statistic"],
        "threshold_statistics_artifact": threshold_config["statistics_artifact"],
        "phase2_total_messages": baseline["total_messages"],
        "phase2_resynchronizations": baseline["resynchronization_count"],
        "centralized_sensor_transmission_reference": len(selected) * len(sensors),
        "message_reduction_vs_phase2_percent": (
            100 * (1 - summary["total_messages"] / baseline["total_messages"])
        ),
        "message_reduction_vs_centralized_percent": (
            100 * (1 - summary["total_messages"] / (len(selected) * len(sensors)))
        ),
        "input_sha256": hashlib.sha256(input_path.read_bytes()).hexdigest(),
        "config_sha256": hashlib.sha256(config_path.read_bytes()).hexdigest(),
        "phase2_baseline_sha256": hashlib.sha256(baseline_path.read_bytes()).hexdigest(),
    })
    events_path = root / "artifacts" / "phase3a_fmic_events.csv"
    summary_path = root / "artifacts" / "phase3a_fmic_summary.json"
    events = pd.DataFrame.from_records(monitor.events)
    events["threshold_source_series"] = threshold_config["source_series"]
    events.to_csv(events_path, index=False)
    summary["events_sha256"] = hashlib.sha256(events_path.read_bytes()).hexdigest()
    summary_path.write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return summary


## Stage 4: Validation, Execution, and Results

The legacy Phase 2-scheduled helper remains for regression coverage only. Normal execution validates the authoritative input and baseline, then runs independent FMIC at alpha 0.10 and writes compact events and summary results.


In [ ]:
# Stage 5: Validate the stable experiment input before monitoring or clustering.
if "build_phase2_trace" not in globals():
    from scripts.phase2_monitoring import build_phase2_trace



def build_phase3a_history_events(
    source_frame,
    threshold,
    *,
    experiment_config,
    alpha_values=None,
    phase2_builder=build_phase2_trace,
):
    """Build compact event rows using the validated experiment configuration."""
    sensor_ids, sensor_columns, _, _, _ = validate_phase3a_config(experiment_config)
    authoritative_threshold = float(experiment_config["threshold"]["p90_value"])
    if float(threshold) != authoritative_threshold:
        raise ValueError("Supplied Phase 3A threshold does not match the authoritative config P90.")
    selected = prepare_phase3a_input(source_frame, experiment_config)
    if alpha_values is None:
        alpha_values = (float(experiment_config["baseline_alpha"]),)
    else:
        alpha_values = tuple(alpha_values)
    if not alpha_values:
        raise ValueError("At least one alpha value is required.")

    selected_times = selected["bucket_time_utc"]
    validation_summary = {
        "row_count": len(selected),
        "first_timestamp": selected_times.iloc[0].isoformat(),
        "last_timestamp": selected_times.iloc[-1].isoformat(),
        "selected_sensor_ids": list(sensor_ids),
        "missing_value_count": int(selected[sensor_columns].isna().sum().sum()),
        "timestamp_gap_count": int(
            selected_times.diff().dt.total_seconds().iloc[1:].ne(
                experiment_config["sampling_interval_seconds"]
            ).sum()
        ),
    }
    print("Phase 3A input validation: " + json.dumps(validation_summary, sort_keys=True))

    # The existing Phase 2 monitor supplies synchronization timing only; this is
    # not a separate Phase 2 comparison run or output artifact.
    baseline = phase2_builder(selected, sensor_columns, threshold)
    if len(baseline) != len(selected):
        raise ValueError("Phase 2 synchronization trace does not match Phase 3A input rows.")
    if not baseline["global_state_evaluable"].astype(bool).all():
        raise ValueError("Phase 3A fixed-cohort input unexpectedly has an undefined global state.")
    if baseline["observed_online_sensor_count"].ne(len(sensor_ids)).any():
        raise ValueError("Phase 3A input violated the fixed six-sensor cohort assumption.")
    if baseline["event_membership_changed"].astype(bool).any():
        raise ValueError("Phase 3A input unexpectedly changed sensor membership.")

    tracker = Phase3AHistoryTracker(sensor_ids)
    event_rows = []
    for row_index, input_row in selected.iterrows():
        baseline_row = baseline.iloc[row_index]
        if int(input_row["bucket_epoch"]) != int(baseline_row["bucket_epoch"]):
            raise ValueError(f"Phase 2/Phase 3A bucket mismatch at row {row_index}.")
        causes = ()
        if int(baseline_row["event_resync_performed"]) == 1:
            causes = tuple(
                cause for cause in str(baseline_row["event_resync_causes"]).split("|") if cause
            )
        histories, causes = tracker.step(
            {
                sensor_id: input_row[f"{sensor_id}_temperature"]
                for sensor_id in sensor_ids
            },
            synchronization_causes=causes,
        )
        if (histories is not None) != (int(baseline_row["event_resync_performed"]) == 1):
            raise ValueError(f"Phase 3A history synchronization mismatch at row {row_index}.")
        if histories is None:
            continue

        normalization = normalize_sensor_histories(histories, sensor_ids)
        alpha_inputs = build_alpha_clustering_inputs(
            tracker.synchronization_id, normalization, alpha_values
        )
        for alpha_input in alpha_inputs:
            evaluation = evaluate_alpha_clustering(alpha_input)
            if evaluation is None:
                raise RuntimeError("A completed stable-cohort event was unexpectedly deferred.")
            partition = evaluation["selected_partition"]
            coordinators = {
                str(cluster_index): max(cluster)
                for cluster_index, cluster in enumerate(partition)
            }
            event_rows.append({
                "clustering_event_id": tracker.synchronization_id,
                "synchronization_id": tracker.synchronization_id,
                "bucket_epoch": int(input_row["bucket_epoch"]),
                "bucket_time_utc": input_row["bucket_time_utc"].isoformat(),
                "synchronization_causes": "|".join(causes),
                "alpha": alpha_input.alpha,
                "raw_sensor_histories": json.dumps(dict(normalization.raw_histories), sort_keys=True),
                "normalized_histories": json.dumps(
                    dict(normalization.normalized_histories)
                    if normalization.normalized_histories is not None else None,
                    sort_keys=True,
                ),
                "normalization_status": normalization.normalization_status,
                "partition_basis": normalization.partition_basis,
                "zero_history_fallback": int(normalization.zero_history_fallback),
                "selected_partition": json.dumps(partition),
                "selected_partition_quality": evaluation["partition_quality"],
                "cluster_count": evaluation["cluster_count"],
                "coordinator_assignments": json.dumps(coordinators, sort_keys=True),
                "coordinator_rule": "lexicographically_highest_member",
                "p90_threshold": authoritative_threshold,
                "threshold_source_series": experiment_config["threshold"]["source_series"],
                "threshold_status": experiment_config["threshold"]["status"],
            })
    return pd.DataFrame.from_records(event_rows), baseline


def build_phase3a_phase2_baseline_summary(trace, experiment_config):
    """Summarize and validate the matched Phase 2 trace for Phase 3A."""
    expected_count = experiment_config["expected_bucket_count"]
    if len(trace) != expected_count:
        raise ValueError("Matched Phase 2 trace has the wrong Phase 3A bucket count.")

    sensor_count = len(experiment_config["selected_sensor_ids"])
    observed_counts = trace["observed_online_sensor_count"]
    if not observed_counts.eq(sensor_count).all():
        raise ValueError("Matched Phase 2 sensor count changed during the stable run.")
    membership_changes = int(trace["event_membership_changed"].sum())
    membership_syncs = int(trace["event_resync_triggered_by_membership_change"].sum())
    if membership_changes or membership_syncs:
        raise ValueError("Matched Phase 2 run unexpectedly reported membership changes.")

    category_columns = (
        "trigger_message_count",
        "request_message_count",
        "response_message_count",
        "broadcast_message_count",
    )
    category_totals = {
        column: int(trace[column].sum()) for column in category_columns
    }
    total_messages = int(trace["total_message_count"].sum())
    if sum(category_totals.values()) != total_messages:
        raise ValueError("Matched Phase 2 communication categories do not reconcile.")

    global_evaluation_count = int(trace["global_state_evaluable"].sum())
    if global_evaluation_count != expected_count:
        raise ValueError("Matched Phase 2 did not evaluate every Phase 3A bucket.")

    threshold = experiment_config["threshold"]
    return {
        "experiment_name": experiment_config["experiment_name"],
        "input_source": experiment_config["input_source"],
        "algorithm": "phase2",
        "selected_sensor_ids": experiment_config["selected_sensor_ids"],
        "start_timestamp_utc": experiment_config["start_timestamp_utc"],
        "end_timestamp_utc": experiment_config["end_timestamp_utc"],
        "endpoint_convention": experiment_config["endpoint_convention"],
        "boundary_convention": experiment_config["boundary_convention"],
        "bucket_count": expected_count,
        "global_evaluation_count": global_evaluation_count,
        "threshold_source_series": threshold["source_series"],
        "threshold_statistic": threshold["statistic"],
        "threshold_value": float(threshold["p90_value"]),
        "total_messages": total_messages,
        "message_category_totals": category_totals,
        "resynchronization_count": int(trace["event_resync_performed"].sum()),
        "membership_change_event_count": membership_changes,
        "membership_triggered_synchronization_count": membership_syncs,
        "observed_sensor_count_min": int(observed_counts.min()),
        "observed_sensor_count_max": int(observed_counts.max()),
        "global_violation_count": int(trace["global_violation"].sum()),
        "centralized_sensor_transmission_reference": expected_count * sensor_count,
        "message_counts_reconcile": True,
        "correctness_status": "passed",
    }


validate_phase3a_config(phase3a_experiment_config)
if globals().get("RUN_PHASE3A_EXECUTION", True):
    phase3a_fmic_summary = run_phase3a_fmic(REPO_ROOT)
    print(json.dumps(phase3a_fmic_summary, indent=2))
